In [1]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import osmnx as ox
import networkx as nx
from shapely.geometry import box
from sklearn.preprocessing import MinMaxScaler

# Create folders
os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

# Bounding box
bbox_polygon = box(120.9850, 14.5750, 121.0150, 14.6100)

print("All imports loaded.")

All imports loaded.


In [2]:
universities = ox.features_from_polygon(
    bbox_polygon,
    tags={"amenity": ["university", "college"]}
)

print(universities[["name", "amenity", "geometry"]].head(20))
print(f"Total universities found: {len(universities)}")

universities.to_file("data/raw/universities.geojson", driver="GeoJSON")
print("Universities saved.")

                                                                   name  \
element  id                                                               
node     255050283       Manila Central University College of Dentistry   
         255068563                            Manila Central University   
         255071563                           De Ocampo Memorial College   
         1732605874                          UST Faculty of Engineering   
         1732605876                 UST College of Fine Arts and Design   
         1974721543               Sudeco Polytechnic Integrated College   
         1974746063                             Access Computer College   
         1974746081                     Montessori Professional College   
         2545181890                         UST College of Architecture   
         4331538009               University of the East College of Law   
         4723440689                         FEU Institute of Technology   
         5205122121      

In [3]:
housing = ox.features_from_polygon(
    bbox_polygon,
    tags={
        "building": ["dormitory", "apartments", "residential"],
        "amenity": "student_accommodation"
    }
)

print(f"Total buildings before filter: {len(housing)}")
print(housing.geometry.geom_type.value_counts())

housing.to_file("data/raw/housing_raw.geojson", driver="GeoJSON")
print("Raw housing saved.")

Total buildings before filter: 313
Polygon    310
Point        3
Name: count, dtype: int64
Raw housing saved.


In [4]:
housing_proj = housing.to_crs(epsg=32651)
universities_proj = universities.to_crs(epsg=32651)

uni_buffer = universities_proj.geometry.union_all().buffer(800)
student_housing = housing_proj[housing_proj.geometry.intersects(uni_buffer)].copy()

print(f"Before filter: {len(housing_proj)}")
print(f"After 800m filter: {len(student_housing)}")  # should be 294

Before filter: 313
After 800m filter: 294


In [5]:
G = ox.graph_from_polygon(bbox_polygon, network_type="drive")
nodes, edges = ox.graph_to_gdfs(G)

print(f"Nodes: {len(nodes)}")
print(f"Road segments: {len(edges)}")

edges.to_file("data/raw/roads.geojson", driver="GeoJSON")
print("Roads saved.")

Nodes: 1487
Road segments: 3599
Roads saved.


In [6]:
waterways = ox.features_from_polygon(
    bbox_polygon,
    tags={"waterway": ["river", "stream", "canal", "drain"]}
)

print(f"Waterway features found: {len(waterways)}")
print(waterways[["name", "waterway"]].dropna(subset=["name"]).head(20))

waterways.to_file("data/raw/waterways.geojson", driver="GeoJSON")
print("Waterways saved.")

Waterway features found: 96
                                        name waterway
element id                                           
way     4304724               Estero de Paco    river
        4443915                  Pasig River    river
        4940654   Estero de Tripa de Gallina    river
        5055435           Estero de Pandacan    river
        5055440           Estero de Pandacan    river
        12805235            Estero de Aviles   stream
        22730074          Estero de Sampaloc    river
        22742600  Estero de Tripa de Gallina    river
        22742601  Estero de Tripa de Gallina    river
        27275946            Estero de Quiapo    river
        27276016           Estero de Uli-Uli   stream
        27276237            Estero de Quiapo    canal
        27276489               Perlita Creek   stream
        27296235                 Pasig River    river
        27323169            Estero de Balete    river
        27323219         Estero de Concordia    river


In [7]:
all_buildings = ox.features_from_polygon(
    bbox_polygon,
    tags={"building": True}
)

print(f"Total buildings in study area: {len(all_buildings)}")

has_levels = all_buildings["building:levels"].notna().sum()
total = len(all_buildings)
print(f"Buildings with levels tag: {has_levels}/{total} ({100*has_levels/total:.1f}%)")

if "building:levels" in student_housing.columns:
    sh_has_levels = student_housing["building:levels"].notna().sum()
    print(f"Student housing with levels: {sh_has_levels}/{len(student_housing)} ({100*sh_has_levels/len(student_housing):.1f}%)")

all_buildings.to_file("data/raw/all_buildings.geojson", driver="GeoJSON")
print("All buildings saved.")

Total buildings in study area: 28892
Buildings with levels tag: 2708/28892 (9.4%)
Student housing with levels: 200/294 (68.0%)
All buildings saved.


In [8]:
student_housing["levels"] = pd.to_numeric(
    student_housing["building:levels"], errors="coerce"
)

print("Levels distribution (before imputation):")
print(student_housing["levels"].describe())
print(f"\nMissing: {student_housing['levels'].isna().sum()}/{len(student_housing)}")

median_levels = student_housing["levels"].median()
student_housing["levels_imputed"] = student_housing["levels"].isna()
student_housing["levels"] = student_housing["levels"].fillna(median_levels)

max_lev = student_housing["levels"].max()
student_housing["vuln_levels"] = 1 - (student_housing["levels"] / max_lev)

print(f"\nMedian levels used for imputation: {median_levels}")
print(f"Imputed rows: {student_housing['levels_imputed'].sum()}")
print(f"Count after levels processing: {len(student_housing)}")  # must be 294

Levels distribution (before imputation):
count    200.00000
mean       7.40000
std        7.10934
min        1.00000
25%        3.00000
50%        5.00000
75%       10.00000
max       52.00000
Name: levels, dtype: float64

Missing: 94/294

Median levels used for imputation: 5.0
Imputed rows: 94
Count after levels processing: 294


In [9]:
waterways_proj = waterways.to_crs(epsg=32651)
waterways_union = waterways_proj.geometry.union_all()

student_housing["dist_waterway_m"] = student_housing.geometry.apply(
    lambda geom: geom.distance(waterways_union)
)

max_dist = student_housing["dist_waterway_m"].max()
student_housing["vuln_waterway"] = 1 - (student_housing["dist_waterway_m"] / max_dist)

print("Waterway distances computed.")
print(student_housing["dist_waterway_m"].describe())
print(f"\nCount after waterway processing: {len(student_housing)}")  # must be 294

Waterway distances computed.
count    294.000000
mean     146.539888
std      182.266312
min        0.000000
25%       36.263751
50%       84.957399
75%      176.619023
max      957.397091
Name: dist_waterway_m, dtype: float64

Count after waterway processing: 294


In [10]:
from shapely.geometry import box as shapely_box

xmin, ymin, xmax, ymax = student_housing.total_bounds
cell_size = 200

cols = np.arange(xmin, xmax, cell_size)
rows = np.arange(ymin, ymax, cell_size)

grid_cells = [
    shapely_box(x, y, x + cell_size, y + cell_size)
    for x in cols for y in rows
]

grid = gpd.GeoDataFrame({'geometry': grid_cells}, crs=student_housing.crs)
grid["cell_id"] = range(len(grid))

# Count buildings per cell
joined = gpd.sjoin(student_housing, grid[["geometry", "cell_id"]], how='left', predicate='intersects')
counts = joined.groupby('cell_id').size().reset_index(name='building_count')
grid = grid.merge(counts, on='cell_id', how='left')
grid['building_count'] = grid['building_count'].fillna(0)

print(f"Grid cells: {len(grid)}")
print(f"Max buildings in a single cell: {grid['building_count'].max()}")
print(f"Cells with at least 1 building: {(grid['building_count'] > 0).sum()}")

grid.to_file("data/processed/density_grid.geojson", driver="GeoJSON")
print("Density grid saved.")

Grid cells: 340
Max buildings in a single cell: 34.0
Cells with at least 1 building: 96
Density grid saved.


In [12]:
# Reset index first to avoid MultiIndex issues
student_housing = student_housing.reset_index(drop=True)

# Find which cell each building falls in
joined_density = gpd.sjoin(
    student_housing[["geometry"]],
    grid[["geometry", "cell_id", "building_count"]],
    how="left",
    predicate="intersects"
)

# Deduplicate in case any building hit multiple cells
joined_density = joined_density[~joined_density.index.duplicated(keep="first")]

# Add building_count to student_housing
student_housing["building_count"] = joined_density["building_count"].values
student_housing["building_count"] = student_housing["building_count"].fillna(0)

print(f"Count after density join: {len(student_housing)}")  # must be 294
print(f"Building count sample:\n{student_housing['building_count'].describe()}")

Count after density join: 294
Building count sample:
count    294.000000
mean      12.278912
std       11.529918
min        1.000000
25%        4.000000
50%        7.000000
75%       21.000000
max       34.000000
Name: building_count, dtype: float64


In [13]:
flood_raw = gpd.read_file("data/Metro Manila/MetroManila_Flood_100year.shp")

flood_raw["hazard_level"] = flood_raw["Var"].map({
    1.0: "low",
    2.0: "medium",
    3.0: "high"
})
flood_raw["hazard_score"] = flood_raw["Var"]

print("Hazard label distribution:")
print(flood_raw[["Var", "hazard_level"]].to_string())

Hazard label distribution:
   Var hazard_level
0  1.0          low
1  2.0       medium
2  3.0         high


In [14]:
flood_proj = flood_raw.to_crs(epsg=32651)

study_bbox = shapely_box(120.9850, 14.5750, 121.0150, 14.6100)
study_bbox_proj = gpd.GeoDataFrame(geometry=[study_bbox], crs="EPSG:4326").to_crs(epsg=32651)

flood_clipped = gpd.clip(flood_proj, study_bbox_proj)

print(f"Features after clipping to study area: {len(flood_clipped)}")
flood_clipped["area_km2"] = flood_clipped.geometry.area / 1_000_000
print("\nArea coverage by hazard level:")
print(flood_clipped.groupby("hazard_level")["area_km2"].sum().round(3))

flood_clipped.to_file("data/processed/flood_hazard.geojson", driver="GeoJSON")
print("\nFlood hazard saved.")

Features after clipping to study area: 3

Area coverage by hazard level:
hazard_level
high      1.020
low       2.713
medium    5.846
Name: area_km2, dtype: float64

Flood hazard saved.


In [15]:
housing_flood = gpd.sjoin(
    student_housing,
    flood_clipped[["hazard_level", "hazard_score", "geometry"]],
    how="left",
    predicate="intersects"
)

print(f"After flood join (before dedup): {len(housing_flood)}")

# Deduplicate using geometry hash
housing_flood["geom_hash"] = housing_flood.geometry.apply(lambda g: hash(g.wkt))
housing_flood = housing_flood.sort_values("hazard_score", ascending=False)
housing_flood = housing_flood.drop_duplicates(subset=["geom_hash"], keep="first")
housing_flood = housing_flood.drop(columns=["geom_hash", "index_right", "cell_id"], errors="ignore")

housing_flood["hazard_level"] = housing_flood["hazard_level"].fillna("none")
housing_flood["hazard_score"] = housing_flood["hazard_score"].fillna(0)
housing_flood["in_flood_zone"] = housing_flood["hazard_score"] > 0

print(f"After dedup: {len(housing_flood)}")  # must be 294
print("\nHazard distribution:")
print(housing_flood["hazard_level"].value_counts())
print(f"\nBuildings in any flood zone: {housing_flood['in_flood_zone'].sum()}/{len(housing_flood)}")

After flood join (before dedup): 454
After dedup: 294

Hazard distribution:
hazard_level
medium    221
low        30
high       23
none       20
Name: count, dtype: int64

Buildings in any flood zone: 274/294


In [16]:
housing_flood.to_file("data/processed/student_housing_final.geojson", driver="GeoJSON")
print("Final clean dataset saved.")

Final clean dataset saved.
